# Implement the classification metrics from scratch

To this exercise you will need to implement the precision, recall, and f1-measure without using scikit-learn or any other library that already implements such metrics.

Your algorithm should take as input the predictions made on the test set (y_pred) and the actual class values of such set (y_test).

You will need to find at least the TP, FP, and FN to compute the three metrics.

You can use this part of code to help your implementation or you can define your own code.

In [1]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
import numpy as np

data = load_breast_cancer()
X, y = data.data, data.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)


def confusion_counts(y_true, y_pred, positive=1):
    """
    Count the TP, FP, FN and TN for a binary problem
    :param y_true: True labels
    :param y_pred: Predicted labels
    :param positive: Label considered as the positive class
    :return: TP, FP, FN, TN
    """
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    TP = np.sum((y_pred == positive) & (y_true == positive))  # predicted positive, really positive
    FP = np.sum((y_pred == positive) & (y_true != positive))  # predicted positive, really negative
    FN = np.sum((y_pred != positive) & (y_true == positive))  # predicted negative, really positive
    TN = np.sum((y_pred != positive) & (y_true != positive))  # predicted negative, really negative
    return TP, FP, FN, TN


def precision(y_true, y_pred):
    """
    Calculate the precision score
    :param y_true: True labels
    :param y_pred: Predicted labels
    :return: Precision score
    """
    TP, FP, FN, TN = confusion_counts(y_true, y_pred)
    # precision = TP / (TP + FP) : among the predicted positives, how many are really positive?
    return TP / (TP + FP) if (TP + FP) > 0 else 0.0


def recall(y_true, y_pred):
    """
    Calculate the recall score
    :param y_true: True labels
    :param y_pred: Predicted labels
    :return: Recall score
    """
    TP, FP, FN, TN = confusion_counts(y_true, y_pred)
    # recall = TP / (TP + FN) : among the real positives, how many did we find?
    return TP / (TP + FN) if (TP + FN) > 0 else 0.0


def f1_score(y_true, y_pred):
    """
    Calculate the F1 score
    :param y_true: True labels
    :param y_pred: Predicted labels
    :return: F1 score
    """
    p = precision(y_true, y_pred)
    r = recall(y_true, y_pred)
    # F1 = harmonic mean of precision and recall
    return 2 * p * r / (p + r) if (p + r) > 0 else 0.0


# Fit a model and make predictions
model = LogisticRegression(max_iter=10000)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

# Calculate precision, recall and F1-score
p = precision(y_test, y_pred)
r = recall(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("Precision: {:.2f}".format(p))
print("Recall: {:.2f}".format(r))
print("F1-Score: {:.2f}".format(f1))

Precision: 0.98
Recall: 0.93
F1-Score: 0.95


### Details of the confusion counts
Positive class = 1 (`benign`) in this dataset.

In [2]:
TP, FP, FN, TN = confusion_counts(y_test, y_pred)
print(f"TP = {TP}, FP = {FP}, FN = {FN}, TN = {TN}  (total = {TP + FP + FN + TN})")
print(f"Precision = {TP}/({TP}+{FP}) = {p:.4f}")
print(f"Recall    = {TP}/({TP}+{FN}) = {r:.4f}")
print(f"F1        = 2*P*R/(P+R) = {f1:.4f}")

TP = 62, FP = 1, FN = 5, TN = 46  (total = 114)
Precision = 62/(62+1) = 0.9841
Recall    = 62/(62+5) = 0.9254
F1        = 2*P*R/(P+R) = 0.9538


### Sanity check with scikit-learn
scikit-learn is used **only** to verify that our from-scratch results are correct.

In [3]:
from sklearn import metrics

print("sklearn precision:", round(metrics.precision_score(y_test, y_pred), 4), "| ours:", round(p, 4))
print("sklearn recall   :", round(metrics.recall_score(y_test, y_pred), 4), "| ours:", round(r, 4))
print("sklearn F1       :", round(metrics.f1_score(y_test, y_pred), 4), "| ours:", round(f1, 4))

sklearn precision: 0.9841 | ours: 0.9841
sklearn recall   : 0.9254 | ours: 0.9254
sklearn F1       : 0.9538 | ours: 0.9538


**Conclusion:** our implementation gives exactly the same values as scikit-learn. Precision tells us how reliable a "benign" prediction is, recall tells us how many benign tumours were found, and the F1-score balances both.